In [1]:
# General packages.
import os
import warnings
import random
import torch
import glob
import time
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter

# Add tools path for additional Python code. 
import sys
sys.path.append("./tools/")

# Text processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Topic modelling
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from top2vec import Top2Vec
from bertopic import BERTopic
from Matave import Matave

In [2]:
# Random States
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)
# Ignore UserWarnings.
warnings.filterwarnings("ignore", category=UserWarning)
# Initialize constant variables.
INPUT_FOLDER = 'data'
# Topic Ranges
K_RANGE = list(range(3, 20))
TOP_N = 10

## Text Processing

In [3]:
stop_words = set(stopwords.words("english"))

In [4]:
# Make function to remove punctuation, make lowercase, remove stopwords, punctuation, remove documents with less than or equal to 1 token.
def preprocessing(notes, min_words=1):
    cleaned_notes = []

    for note in notes:
        tokens = word_tokenize(note, language='english')
        tokens = [token.lower() for token in tokens]
        tokens = [token for token in tokens if token.isalpha() and token not in stop_words]

        if len(tokens) >= min_words:
            cleaned_notes.append(" ".join(tokens))

    return cleaned_notes

In [5]:
all_text_notes = []
for file in os.listdir('./syntheticData'):
    if '.csv' in file:
        all_text_notes.extend(preprocessing(pd.read_csv(f"./syntheticData/{file}")['report'].values.tolist()))

In [6]:
len(all_text_notes)

5783

## Topic Modelling

In [7]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

### LDA

In [8]:
def lda_analysis(tokenized_texts, dictionary, corpus, dataset_name):
    metric_results = {'Dataset Name': [], 'Algorithm Name': [], 'Coherence': [], 'Diversity': [], 'Redundancy': [], 'Time': [], 'Top Topic Words': []}
    
    for k in K_RANGE:
        start = time.time()

        lda_model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=RANDOM_STATE,
            passes=10
        )

        lda_topics = [
            [word for word, _ in lda_model.show_topic(i, topn=TOP_N)]
            for i in range(k)
        ]

        end = time.time()

        metric_results['Algorithm Name'].append(f'LDA (K={k})')
        metric_results['Dataset Name'].append(dataset_name)
        metric_results['Coherence'].append(
            get_coherence_score(lda_topics, tokenized_texts, dictionary, 'c_v')
        )
        metric_results['Diversity'].append(get_diversity_score(lda_topics))
        metric_results['Redundancy'].append(compute_topic_redundancy(lda_topics))
        metric_results['Time'].append(end - start)
        metric_results['Top Topic Words'].append(lda_topics)

    def normalize(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-9)

    norm_coh = normalize(np.array(metric_results['Coherence']))
    norm_div = normalize(np.array(metric_results['Diversity']))
    norm_red = normalize(np.array(metric_results['Redundancy']))

    w_coh = 1
    w_div = 1
    w_red = 1

    composite_scores = (
        w_coh * norm_coh +
        w_div * norm_div +
        w_red * norm_red
    )

    best_index = np.argmax(composite_scores)

    print(f"Coherence: {metric_results['Coherence'][best_index]}")
    print(f"Diversity: {metric_results['Diversity'][best_index]}")
    print(f"Inverse Redundancy: {metric_results['Redundancy'][best_index]}")
    print(f"Time (seconds): {metric_results['Time'][best_index]}")

    temp_top_row = metric_results['Top Topic Words'][best_index]
    print("----- Cluster Topics -----")
    for topic in temp_top_row:
        print(topic)
    print(f"Number of Topics: {len(temp_top_row)}")
    return best_index, metric_results

### NMF

In [9]:
def nmf_analysis(texts, tokenized_texts, dictionary, dataset_name):
    # Vectorize texts for NMF.
    vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=2,
    stop_words='english'
    )
    tfidf = vectorizer.fit_transform(texts)
    feature_names = vectorizer.get_feature_names_out()

    metric_results = {'Dataset Name': [], 'Algorithm Name': [], 'Coherence': [], 'Diversity': [], 'Redundancy': [], 'Time': [], 'Top Topic Words': []}
    for k in K_RANGE:
        start = time.time()

        nmf_model = NMF(
            n_components=k,
            random_state=RANDOM_STATE
        )
        nmf_model.fit(tfidf)

        nmf_topics = [
            [feature_names[i] for i in topic.argsort()[:-TOP_N - 1:-1]]
            for topic in nmf_model.components_
        ]

        end = time.time()

        metric_results['Algorithm Name'].append(f'NMF (K={k})')
        metric_results['Dataset Name'].append(dataset_name)
        metric_results['Coherence'].append(
            get_coherence_score(nmf_topics, tokenized_texts, dictionary, 'c_v')
        )
        metric_results['Diversity'].append(get_diversity_score(nmf_topics))
        metric_results['Redundancy'].append(compute_topic_redundancy(nmf_topics))
        metric_results['Time'].append(end - start)
        metric_results['Top Topic Words'].append(nmf_topics)

    def normalize(x):
            return (x - x.min()) / (x.max() - x.min() + 1e-9)

    norm_coh = normalize(np.array(metric_results['Coherence']))
    norm_div = normalize(np.array(metric_results['Diversity']))
    norm_red = normalize(np.array(metric_results['Redundancy']))

    w_coh = 1
    w_div = 1
    w_red = 1

    composite_scores = (
        w_coh * norm_coh +
        w_div * norm_div +
        w_red * norm_red
    )

    best_index = np.argmax(composite_scores)

    print(f"Coherence: {metric_results['Coherence'][best_index]}")
    print(f"Diversity: {metric_results['Diversity'][best_index]}")
    print(f"Inverse Redundancy: {metric_results['Redundancy'][best_index]}")
    print(f"Time (seconds): {metric_results['Time'][best_index]}")

    temp_top_row = metric_results['Top Topic Words'][best_index]
    print("----- Cluster Topics -----")
    for topic in temp_top_row:
        print(topic)
    print(f"Number of Topics: {len(temp_top_row)}")
    return best_index, metric_results

### Top2Vec

In [10]:
def top2vec_analysis(texts, tokenized_texts, dictionary):
    start = time.time()
    top2vec_model = Top2Vec(
        texts,
        embedding_model='all-MiniLM-L6-v2',
        speed="learn"
    )
    cluster_topics = (top2vec_model.get_topics())[0]
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)

    print(f"Number of Topics: {len(cluster_topics)}")

### BERTopic

In [11]:
def bertopic_analysis(texts, tokenized_texts, dictionary):
    start = time.time() 
    topic_model = BERTopic(
        embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    )
    _topics, _probs = topic_model.fit_transform(texts)
    cluster_topics = list(topic_model.get_topic_info()['Representation'])
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)
        
    print(f"Number of Topics: {len(cluster_topics)}")

### MATAVE

In [12]:
def matave_analysis(texts, tokenized_texts, dictionary, file_name = ""):
    # MATAVE
    start = time.time()
    matave = Matave(texts)
    matave.fit(k_range = K_RANGE)
    cluster_topics = [topic.split() for topic in matave.top_topic_words.values()]
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)

    print(f"Number of Topics: {len(cluster_topics)}")

In [13]:
# Prepare components for evaluation.
tokenized_all_notes = [word_tokenize(text.lower()) for text in all_text_notes]
dictionary_all_notes = Dictionary(tokenized_all_notes)
corpus_all_notes = [dictionary_all_notes.doc2bow(text) for text in tokenized_all_notes]

# lda
print("----------- LDA -----------")
lda_analysis(tokenized_all_notes, dictionary_all_notes, corpus_all_notes, "All Notes")
# nmf
print("----------- NMF -----------")
nmf_analysis(all_text_notes, tokenized_all_notes, dictionary_all_notes, "All Notes")
# top2vec
print("----------- Top2Vec -----------")
top2vec_analysis(all_text_notes, tokenized_all_notes, dictionary_all_notes)
# bertopic
print("----------- BERTopic -----------")
bertopic_analysis(all_text_notes, tokenized_all_notes, dictionary_all_notes)
# matave
print("----------- MATAVE -----------")
matave_analysis(all_text_notes, tokenized_all_notes, dictionary_all_notes)

----------- LDA -----------
Coherence: 0.7147790393144048
Diversity: 0.9
Inverse Redundancy: 0.96
Time (seconds): 5.6747472286224365
----- Cluster Topics -----
['care', 'family', 'plan', 'skin', 'today', 'intact', 'regular', 'updated', 'requested', 'conference']
['experiencing', 'resident', 'nausea', 'difficulty', 'breath', 'experienced', 'mobility', 'episodes', 'symptoms', 'frequent']
['skin', 'fall', 'experienced', 'staff', 'integrity', 'incident', 'help', 'wheelchair', 'bed', 'minor']
['pain', 'medication', 'relief', 'back', 'discomfort', 'management', 'requested', 'complained', 'reported', 'night']
['signs', 'care', 'palliative', 'confusion', 'resident', 'support', 'restlessness', 'team', 'agitation', 'showed']
['afternoon', 'enjoyed', 'visit', 'day', 'morning', 'spent', 'today', 'room', 'time', 'needed']
Number of Topics: 6
----------- NMF -----------


2026-02-25 09:25:31,506 - top2vec - INFO - Pre-processing documents for training
2026-02-25 09:25:31,614 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Coherence: 0.7492812508017148
Diversity: 1.0
Inverse Redundancy: 1.0
Time (seconds): 0.037009239196777344
----- Cluster Topics -----
['pain', 'medication', 'relief', 'complained', 'management', 'discomfort', 'severe', 'requested', 'given', 'persistent']
['signs', 'resident', 'confusion', 'restlessness', 'support', 'agitation', 'night', 'requiring', 'showed', 'showing']
['care', 'skin', 'plan', 'family', 'palliative', 'intact', 'conference', 'turning', 'regular', 'goals']
Number of Topics: 3
----------- Top2Vec -----------


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-25 09:25:33,511 - top2vec - INFO - Creating joint document/word embedding
2026-02-25 09:25:39,984 - top2vec - INFO - Creating lower dimension embedding of documents
2026-02-25 09:25:50,019 - top2vec - INFO - Finding dense areas of documents
2026-02-25 09:25:50,121 - top2vec - INFO - Finding topics


...

Coherence: 0.5697887319987185
Diversity: 0.6130841121495327
Inverse Redundancy: 0.9837065773232234
Time (seconds): 9.016989946365356
----- Cluster Topics -----
['pain', 'medication', 'night', 'enjoyed', 'back', 'afternoon', 'morning', 'nausea', 'restlessness', 'feeling']
['breath', 'tightness', 'oxygen', 'respiratory', 'shortness', 'chest', 'breathing', 'distress', 'therapy', 'difficulty']
['care', 'requested', 'discussion', 'needs', 'family', 'meeting', 'updates', 'plan', 'palliative', 'preferences']
['itching', 'redness', 'skin', 'irritation', 'peeling', 'applied', 'cream', 'lotion', 'soothing', 'reaction']
['grandchildren', 'visit', 'brought', 'call', 'visited', 'phone', 'flowers', 'spirits', 'laughter', 'today']
['apathy', 'activities', 'disinterest', 'engagement', 'social', 'interest', 'withdrawal', 'interactions', 'mood', 'lack']
['diarrhea', 'dehydration', 'hydration', 'diarrhoea', 'electrolyte', 'episodes', 'abdominal', 'bout', 'fluid', 'frequent']
['together', 'lunch', 'visite

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/91 [00:00<?, ?it/s]

Coherence: 0.430513681993706
Diversity: 0.85
Inverse Redundancy: 0.9571428571428572
Time (seconds): 21.43278217315674
----- Cluster Topics -----
['disorientation', 'disinterest', 'cognitive', 'withdrawal', 'mental', 'apathy', 'depression', 'isolation', 'forgetfulness', 'social']
['itching', 'ulcer', 'redness', 'end', 'sacrum', 'heels', 'irritation', 'conference', 'present', 'heel']
['vomiting', 'swallowing', 'antiemetic', 'incontinence', 'nausea', 'food', 'diarrhea', 'diet', 'speech', 'dehydration']
['joints', 'analgesia', 'end', 'heating', 'pad', 'conference', 'meeting', 'present', 'massage', 'heat']
['unrelenting', 'hopeful', 'cancer', 'hold', 'uncomfortable', 'overnight', 'temporary', 'despite', 'coming', 'effect']
['shortness', 'incontinence', 'oxygen', 'respiratory', 'nails', 'breathing', 'chest', 'urinary', 'breath', 'trimmed']
['injuries', 'incident', 'transfer', 'wheelchair', 'falls', 'prevention', 'trying', 'attempting', 'sustained', 'weakness']
['shortness', 'oxygen', 'respir